# 03 - Gold: Delta Silver para PostgreSQL

Este notebook gera tabelas Gold a partir da camada Silver Delta e publica os resultados em PostgreSQL.

Variaveis de ambiente esperadas:

- `AIDP_CATALOG`, `SILVER_SCHEMA`
- `PG_GOLD_HOST`, `PG_GOLD_PORT`, `PG_GOLD_DB`, `PG_GOLD_SCHEMA`, `PG_GOLD_USER`, `PG_GOLD_PASSWORD`
- Se as variaveis `PG_GOLD_*` nao existirem, o notebook usa `PG_*` como fallback.
- `PG_GOLD_SSLMODE` opcional, padrao `require`
- `GOLD_POSTGRES_WRITE_MODE`, padrao `overwrite`

### Proxima celula
Importa bibliotecas e funcoes PySpark usadas para agregacoes, janelas analiticas e acesso JDBC ao PostgreSQL Gold.


In [ ]:
import os
import urllib.request
from pyspark.sql import Window
from pyspark.sql import functions as F

### Proxima celula
Define parametros de origem Silver, conexao PostgreSQL Gold, modo de escrita e limite usado para classificar estoque baixo.


In [ ]:
CATALOG = os.getenv('AIDP_CATALOG', 'main')
SILVER_SCHEMA = os.getenv('SILVER_SCHEMA', 'silver')

PG_GOLD_HOST = os.getenv('PG_GOLD_HOST', os.getenv('PG_HOST'))
PG_GOLD_PORT = os.getenv('PG_GOLD_PORT', os.getenv('PG_PORT', '5432'))
PG_GOLD_DB = os.getenv('PG_GOLD_DB', os.getenv('PG_DB'))
PG_GOLD_SCHEMA = os.getenv('PG_GOLD_SCHEMA', 'gold')
PG_GOLD_USER = os.getenv('PG_GOLD_USER', os.getenv('PG_USER'))
PG_GOLD_PASSWORD = os.getenv('PG_GOLD_PASSWORD', os.getenv('PG_PASSWORD'))
PG_GOLD_SSLMODE = os.getenv('PG_GOLD_SSLMODE', os.getenv('PG_SSLMODE', 'require'))
GOLD_POSTGRES_WRITE_MODE = os.getenv('GOLD_POSTGRES_WRITE_MODE', 'overwrite')
LOW_STOCK_THRESHOLD = int(os.getenv('LOW_STOCK_THRESHOLD', '10'))

required_gold_vars = {
    'PG_GOLD_HOST': PG_GOLD_HOST,
    'PG_GOLD_DB': PG_GOLD_DB,
    'PG_GOLD_USER': PG_GOLD_USER,
    'PG_GOLD_PASSWORD': PG_GOLD_PASSWORD,
}
missing = [name for name, value in required_gold_vars.items() if not value]
if missing:
    raise ValueError(f'Missing PostgreSQL Gold configuration: {missing}')

### Proxima celula
Cria funcoes auxiliares para ler tabelas Silver, qualificar nomes no catalogo AIDP e montar nomes seguros no PostgreSQL.


In [ ]:
def q_table(catalog, schema, table):
    return f'`{catalog}`.`{schema}`.`{table}`'


def silver(table):
    return spark.table(q_table(CATALOG, SILVER_SCHEMA, table))


def quote_pg_identifier(value):
    return chr(34) + value.replace(chr(34), chr(34) + chr(34)) + chr(34)


def pg_table(schema, table):
    return f'{quote_pg_identifier(schema)}.{quote_pg_identifier(table)}'

### Proxima celula
Baixa, registra e distribui o driver JDBC do PostgreSQL para publicar as tabelas Gold com suporte a SSL.


In [ ]:
JDBC_JAR_PATH = os.getenv('POSTGRES_JDBC_JAR_PATH', '/tmp/postgresql-42.7.4.jar')
JDBC_JAR_URL = os.getenv(
    'POSTGRES_JDBC_JAR_URL',
    'https://repo1.maven.org/maven2/org/postgresql/postgresql/42.7.4/postgresql-42.7.4.jar',
)

try:
    spark._jvm.java.lang.Class.forName('org.postgresql.Driver')
    print('PostgreSQL JDBC driver already available on cluster classpath')
except Exception:
    if not os.path.exists(JDBC_JAR_PATH):
        urllib.request.urlretrieve(JDBC_JAR_URL, JDBC_JAR_PATH)

    gw = spark._sc._gateway
    jar_url = spark._jvm.java.io.File(JDBC_JAR_PATH).toURI().toURL()
    url_array = gw.new_array(spark._jvm.java.net.URL, 1)
    url_array[0] = jar_url
    class_loader = spark._jvm.java.net.URLClassLoader(
        url_array,
        spark._jvm.java.lang.ClassLoader.getSystemClassLoader(),
    )
    spark._jvm.java.lang.Thread.currentThread().setContextClassLoader(class_loader)
    driver_class = spark._jvm.java.lang.Class.forName('org.postgresql.Driver', True, class_loader)
    spark._jvm.java.sql.DriverManager.registerDriver(driver_class.newInstance())
    spark._jsc.addJar(JDBC_JAR_PATH)

jdbc_params = f'?sslmode={PG_GOLD_SSLMODE}' if PG_GOLD_SSLMODE else ''
JDBC_URL = f'jdbc:postgresql://{PG_GOLD_HOST}:{PG_GOLD_PORT}/{PG_GOLD_DB}{jdbc_params}'

### Proxima celula
Define funcoes para criar o schema Gold no PostgreSQL e gravar cada DataFrame agregado como tabela PostgreSQL.


In [ ]:
def ensure_postgres_schema(schema):
    props = spark._jvm.java.util.Properties()
    props.setProperty('user', PG_GOLD_USER)
    props.setProperty('password', PG_GOLD_PASSWORD)
    conn = spark._jvm.java.sql.DriverManager.getConnection(JDBC_URL, props)
    try:
        stmt = conn.createStatement()
        try:
            stmt.execute(f'CREATE SCHEMA IF NOT EXISTS {quote_pg_identifier(schema)}')
        finally:
            stmt.close()
    finally:
        conn.close()


def write_gold_postgres(df, table):
    (
        df.write
        .format('jdbc')
        .option('url', JDBC_URL)
        .option('driver', 'org.postgresql.Driver')
        .option('user', PG_GOLD_USER)
        .option('password', PG_GOLD_PASSWORD)
        .option('dbtable', pg_table(PG_GOLD_SCHEMA, table))
        .mode(GOLD_POSTGRES_WRITE_MODE)
        .save()
    )
    print(f'Wrote PostgreSQL table {PG_GOLD_SCHEMA}.{table}')

### Proxima celula
Carrega as tabelas Silver necessarias e garante que o schema Gold exista no PostgreSQL antes das escritas.


In [ ]:
dim_location = silver('dim_location')
dim_customer = silver('dim_customer')
dim_item = silver('dim_item')
fact_inventory = silver('fact_inventory')
fact_order_header = silver('fact_order_header')
fact_order_line = silver('fact_order_line')
fact_payment = silver('fact_payment')
fact_pending_order = silver('fact_pending_order')

ensure_postgres_schema(PG_GOLD_SCHEMA)

### Proxima celula
Gera gold_sales_summary com pedidos, quantidade, receita e ticket medio por data, armazem e distrito.


In [ ]:
order_line_with_location = (
    fact_order_line.alias('ol')
    .join(
        dim_location.alias('loc'),
        (F.col('ol.warehouse_id') == F.col('loc.warehouse_id'))
        & (F.col('ol.district_id') == F.col('loc.district_id')),
        'left',
    )
    .select(
        F.col('ol.order_date').alias('sales_date'),
        F.col('ol.warehouse_id').alias('warehouse_id'),
        F.col('loc.warehouse_name').alias('warehouse_name'),
        F.col('ol.district_id').alias('district_id'),
        F.col('loc.district_name').alias('district_name'),
        F.concat_ws('-', F.col('ol.warehouse_id'), F.col('ol.district_id'), F.col('ol.order_id')).alias('order_key'),
        F.col('ol.quantity').alias('quantity'),
        F.col('ol.amount').alias('amount'),
    )
)

gold_sales_summary = (
    order_line_with_location
    .groupBy('sales_date', 'warehouse_id', 'warehouse_name', 'district_id', 'district_name')
    .agg(
        F.countDistinct('order_key').alias('total_orders'),
        F.sum('quantity').cast('long').alias('total_quantity'),
        F.sum('amount').cast('decimal(18,2)').alias('total_revenue'),
    )
    .withColumn(
        'average_ticket',
        F.when(F.col('total_orders') > 0, F.col('total_revenue') / F.col('total_orders')).cast('decimal(18,2)'),
    )
)

write_gold_postgres(gold_sales_summary, 'gold_sales_summary')

### Proxima celula
Gera gold_customer_360 consolidando cadastro, pedidos, compras e pagamentos por cliente.


In [ ]:
order_header_agg = (
    fact_order_header
    .groupBy('warehouse_id', 'district_id', 'customer_id')
    .agg(
        F.countDistinct(F.concat_ws('-', 'warehouse_id', 'district_id', 'order_id')).alias('total_orders'),
        F.min('order_date').alias('first_order_date'),
        F.max('order_date').alias('last_order_date'),
    )
)

purchase_agg = (
    fact_order_line
    .groupBy('warehouse_id', 'district_id', 'customer_id')
    .agg(
        F.sum('amount').cast('decimal(18,2)').alias('total_purchased_amount'),
        F.sum('quantity').cast('long').alias('total_items_purchased'),
    )
)

payment_agg = (
    fact_payment
    .groupBy(
        F.col('customer_warehouse_id').alias('warehouse_id'),
        F.col('customer_district_id').alias('district_id'),
        'customer_id',
    )
    .agg(
        F.sum('payment_amount').cast('decimal(18,2)').alias('total_paid_amount'),
        F.max('payment_date').alias('last_payment_date'),
    )
)

gold_customer_360 = (
    dim_customer.alias('c')
    .join(order_header_agg.alias('o'), ['warehouse_id', 'district_id', 'customer_id'], 'left')
    .join(purchase_agg.alias('p'), ['warehouse_id', 'district_id', 'customer_id'], 'left')
    .join(payment_agg.alias('pay'), ['warehouse_id', 'district_id', 'customer_id'], 'left')
    .select(
        'warehouse_id',
        'district_id',
        'customer_id',
        'customer_name',
        'city',
        'state',
        'credit_status',
        'credit_limit',
        'discount',
        'balance',
        'ytd_payment',
        'payment_count',
        'delivery_count',
        F.coalesce(F.col('total_orders'), F.lit(0)).alias('total_orders'),
        F.coalesce(F.col('total_purchased_amount'), F.lit(0)).cast('decimal(18,2)').alias('total_purchased_amount'),
        F.coalesce(F.col('total_paid_amount'), F.lit(0)).cast('decimal(18,2)').alias('total_paid_amount'),
        'total_items_purchased',
        'first_order_date',
        'last_order_date',
        'last_payment_date',
    )
)

write_gold_postgres(gold_customer_360, 'gold_customer_360')

### Proxima celula
Gera gold_inventory_health com status de estoque, quantidade, valor em inventario e movimentacao por item e armazem.


In [ ]:
gold_inventory_health = (
    fact_inventory
    .withColumn(
        'stock_status',
        F.when(F.col('stock_quantity') <= 0, F.lit('OUT_OF_STOCK'))
        .when(F.col('stock_quantity') <= LOW_STOCK_THRESHOLD, F.lit('LOW_STOCK'))
        .otherwise(F.lit('OK')),
    )
    .select(
        'warehouse_id',
        'item_id',
        'item_name',
        'item_price',
        'stock_quantity',
        'stock_ytd',
        'order_count',
        'remote_count',
        'inventory_value',
        'stock_status',
    )
)

write_gold_postgres(gold_inventory_health, 'gold_inventory_health')

### Proxima celula
Gera gold_order_fulfillment com indicadores de pedidos pendentes, entregues, sem transportadora e prazo medio.


In [ ]:
line_delivery_agg = (
    fact_order_line
    .groupBy('warehouse_id', 'district_id', 'order_id')
    .agg(
        F.max('delivery_timestamp').alias('last_delivery_timestamp'),
        F.sum('amount').cast('decimal(18,2)').alias('order_amount'),
    )
)

pending_keys = fact_pending_order.select('warehouse_id', 'district_id', 'order_id').withColumn('is_pending', F.lit(1))

fulfillment_base = (
    fact_order_header.alias('o')
    .join(line_delivery_agg.alias('l'), ['warehouse_id', 'district_id', 'order_id'], 'left')
    .join(pending_keys.alias('p'), ['warehouse_id', 'district_id', 'order_id'], 'left')
)

gold_order_fulfillment = (
    fulfillment_base
    .groupBy('order_date', 'warehouse_id', 'district_id')
    .agg(
        F.countDistinct(F.concat_ws('-', 'warehouse_id', 'district_id', 'order_id')).alias('total_orders'),
        F.sum(F.coalesce(F.col('is_pending'), F.lit(0))).cast('long').alias('pending_orders'),
        F.sum(F.when(F.col('last_delivery_timestamp').isNotNull(), F.lit(1)).otherwise(F.lit(0))).cast('long').alias('delivered_orders'),
        F.sum(F.when(F.col('carrier_id').isNull(), F.lit(1)).otherwise(F.lit(0))).cast('long').alias('orders_without_carrier'),
        F.avg(F.col('all_local_flag')).cast('decimal(10,4)').alias('all_local_ratio'),
        F.avg(F.datediff(F.to_date('last_delivery_timestamp'), F.col('order_date'))).cast('decimal(18,2)').alias('avg_delivery_days'),
    )
)

write_gold_postgres(gold_order_fulfillment, 'gold_order_fulfillment')

### Proxima celula
Gera gold_product_performance com receita, quantidade, pedidos por item e rankings de desempenho.


In [ ]:
product_agg = (
    fact_order_line
    .withColumn('order_key', F.concat_ws('-', 'warehouse_id', 'district_id', 'order_id'))
    .groupBy('item_id', 'item_name')
    .agg(
        F.sum('amount').cast('decimal(18,2)').alias('total_revenue'),
        F.sum('quantity').cast('long').alias('total_quantity'),
        F.countDistinct('order_key').alias('orders_with_item'),
        F.avg('unit_price').cast('decimal(18,2)').alias('average_unit_price'),
    )
)

gold_product_performance = (
    product_agg
    .withColumn('revenue_rank', F.dense_rank().over(Window.orderBy(F.col('total_revenue').desc_nulls_last())))
    .withColumn('quantity_rank', F.dense_rank().over(Window.orderBy(F.col('total_quantity').desc_nulls_last())))
)

write_gold_postgres(gold_product_performance, 'gold_product_performance')

### Proxima celula
Gera gold_financial_kpis consolidando receita estimada, pagamentos, impostos e valores acumulados por data e localidade.


In [ ]:
payment_finance = (
    fact_payment
    .groupBy(
        F.col('payment_date').alias('metric_date'),
        'warehouse_id',
        'district_id',
    )
    .agg(
        F.countDistinct('payment_id').alias('total_payments'),
        F.sum('payment_amount').cast('decimal(18,2)').alias('total_payment_amount'),
    )
)

order_finance = (
    fact_order_line
    .groupBy(
        F.col('order_date').alias('metric_date'),
        'warehouse_id',
        'district_id',
    )
    .agg(
        F.sum('amount').cast('decimal(18,2)').alias('estimated_revenue'),
        F.countDistinct(F.concat_ws('-', 'warehouse_id', 'district_id', 'order_id')).alias('total_orders'),
    )
)

gold_financial_kpis = (
    order_finance.alias('o')
    .join(payment_finance.alias('p'), ['metric_date', 'warehouse_id', 'district_id'], 'full')
    .join(dim_location.alias('loc'), ['warehouse_id', 'district_id'], 'left')
    .select(
        'metric_date',
        'warehouse_id',
        'warehouse_name',
        'district_id',
        'district_name',
        F.coalesce(F.col('total_orders'), F.lit(0)).alias('total_orders'),
        F.coalesce(F.col('estimated_revenue'), F.lit(0)).cast('decimal(18,2)').alias('estimated_revenue'),
        F.coalesce(F.col('total_payments'), F.lit(0)).alias('total_payments'),
        F.coalesce(F.col('total_payment_amount'), F.lit(0)).cast('decimal(18,2)').alias('total_payment_amount'),
        F.col('warehouse_ytd'),
        F.col('district_ytd'),
        F.col('warehouse_tax'),
        F.col('district_tax'),
    )
)

write_gold_postgres(gold_financial_kpis, 'gold_financial_kpis')

### Proxima celula
Monta um payload JSON de sucesso para uso em workflows AIDP ou para exibicao quando executado interativamente.


In [ ]:
import json

payload = {
    'status': 'SUCCESS',
    'layer': 'gold',
    'postgres_schema': PG_GOLD_SCHEMA,
    'tables': [
        'gold_sales_summary',
        'gold_customer_360',
        'gold_inventory_health',
        'gold_order_fulfillment',
        'gold_product_performance',
        'gold_financial_kpis',
    ],
}

try:
    oidlUtils.notebook.exit(json.dumps(payload))
except NameError:
    print(json.dumps(payload))